### Building a chatbot with conversation history

This chatbot will be able to have a conversation and remember previous interactions.

In [12]:
import os
from dotenv import load_dotenv

from langchain.chat_models import init_chat_model

from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory


load_dotenv()

True

##### Model without conversation history

In [7]:
model = init_chat_model(model="groq:llama-3.1-8b-instant")
model

ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 8192, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x0000012491FCE960>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000012491FCDBB0>, model_name='llama-3.1-8b-instant', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [9]:
from langchain_core.messages import HumanMessage, BaseMessage, AIMessage

# input to model should be list of basemessage or prompt

user_message = [HumanMessage(content="Hi, Im Mounica. Currently learning langchain and langgraph")]

model.invoke(user_message)

AIMessage(content="Nice to meet you, Mounica. Langchain and Langgraph are exciting areas of research and development in natural language processing (NLP). Langchain refers to a type of AI that integrates multiple language models to create a more comprehensive and coherent understanding of language, while Langgraph is a framework for building graph-based language models.\n\nWhat are your goals in learning langchain and langgraph? Are you looking to apply these concepts in a specific project or industry, or are you simply interested in the theoretical aspects of NLP?\n\nAlso, what's your background in computer science and NLP? Have you worked with any other NLP libraries or frameworks, such as BERT, RoBERTa, or Transformers?", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 143, 'prompt_tokens': 49, 'total_tokens': 192, 'completion_time': 0.232088592, 'completion_tokens_details': None, 'prompt_time': 0.002393447, 'prompt_tokens_details': None, 'queue_time': 

In [10]:
user_message = [HumanMessage(content="Hi, Im Mounica. Currently learning langchain and langgraph"),
                AIMessage(content="Hello Mounica. Langchain and Langgraph are exciting topics in the field of natural language processing (NLP). Langchain focuses on creating a framework for building large language models and developing applications that utilize them, while Langgraph is a type of graph-based language model that aims to better represent complex linguistic structures.\n\nWhat specific aspects of Langchain and Langgraph are you interested in learning more about? Are you looking to build a project or implement these concepts in a particular application? I'd be happy to help you with any questions or provide guidance on getting started."),
                HumanMessage(content="do you remember my name?")]
model.invoke(user_message)

AIMessage(content="You're Mounica. I'll try to remember it for our conversation.", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 17, 'prompt_tokens': 178, 'total_tokens': 195, 'completion_time': 0.035578157, 'completion_tokens_details': None, 'prompt_time': 0.010055469, 'prompt_tokens_details': None, 'queue_time': 0.0552181, 'total_time': 0.045633626}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019b7bd8-40a6-7091-8c4b-4725c6ebf335-0', usage_metadata={'input_tokens': 178, 'output_tokens': 17, 'total_tokens': 195})

In [11]:
model.invoke([HumanMessage(content="do you still remember my name?")])

AIMessage(content="I'm a large language model, I don't have personal memories or the ability to recall previous conversations or users' names. Each time you interact with me, it's a new conversation, and I don't retain any information from previous chats.\n\nSo, I don't remember your name, but I'm happy to chat with you and help with any questions or topics you'd like to discuss!", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 80, 'prompt_tokens': 42, 'total_tokens': 122, 'completion_time': 0.098876778, 'completion_tokens_details': None, 'prompt_time': 0.001931141, 'prompt_tokens_details': None, 'queue_time': 0.050241819, 'total_time': 0.100807919}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019b7bd9-389a-78a3-bcd5-bf8e90cddbab-0', usage_metadata={'input_tokens': 42, 'output_tokens': 80, 'total_tokens': 122})

##### Model with conversation history

We can use a Message History class to wrap our model and make it stateful.This will keep track of inputs and outputs of the model and store them in some datastore. Future interactions will then load these messages and pass them into the chain as part of the input.

In [13]:
session_store={}
def get_session_history(session_id:str)->BaseChatMessageHistory:
    "this function gets the message history for the given session_id"
    if session_id not in session_store.keys():
        session_store[session_id] = ChatMessageHistory()
    return session_store[session_id]

with_message_history = RunnableWithMessageHistory(model, get_session_history)    

In [15]:
config={"configurable":{"session_id":"user_1"}}

response = with_message_history.invoke(user_message, config=config)
response

AIMessage(content='Your name is Mounica.', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 8, 'prompt_tokens': 178, 'total_tokens': 186, 'completion_time': 0.009852717, 'completion_tokens_details': None, 'prompt_time': 0.011452128, 'prompt_tokens_details': None, 'queue_time': 0.054969152, 'total_time': 0.021304845}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_ff2b098aaf', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019b7be5-5d83-76c1-a495-cfd5bd4b7af1-0', usage_metadata={'input_tokens': 178, 'output_tokens': 8, 'total_tokens': 186})

In [16]:
query="do you still remember me?"
response_b = with_message_history.invoke(query, config=config)
response_b


AIMessage(content='Yes, I still remember your name as Mounica. It was nice chatting with you earlier about Langchain and Langgraph. How can I assist you further today?', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 35, 'prompt_tokens': 201, 'total_tokens': 236, 'completion_time': 0.048613893, 'completion_tokens_details': None, 'prompt_time': 0.013000182, 'prompt_tokens_details': None, 'queue_time': 0.055207858, 'total_time': 0.061614075}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_1151d4f23c', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019b7be6-35fd-72b0-b3fa-e1a51a0637dc-0', usage_metadata={'input_tokens': 201, 'output_tokens': 35, 'total_tokens': 236})